In [1]:
from numpy import array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
import numpy as np
import pandas as pd
import tensorflow as tf

In [2]:
# split a univariate sequence into samples
def split_sequence(sequence, n_steps):
    X, y = list(), list()
    for i in range(len(sequence)):
        # find the end of this pattern
        end_ix = i + n_steps
        # check if we are beyond the sequence
        if end_ix > len(sequence)-1:
            break
        # gather input and output parts of the pattern
        seq_x, seq_y = sequence[i:end_ix], sequence[end_ix]
        X.append(seq_x)
        y.append(seq_y)
    return array(X), array(y)

In [3]:
# define input sequence
parser = (lambda x:datetime.datetime.strptime(x, '%Y.%m.%d')) 
df = pd.read_csv('sp_beaches_update.csv', parse_dates=['Date'])
df = df.sort_values(by=['Date'])
df=df.loc[~df['Enterococcus'].isnull()]
#remover a praia do Leste, da cidade de iguape, pois esta praia sumiu por erosão em 2012
#remover a Lagoa Prumirim, da cidade de Ubatuba, pois esta praia possui somente 3 medições
df = df.loc[df['Beach']!='DO LESTE'].loc[df['Beach']!='LAGOA PRUMIRIM']
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 69016 entries, 0 to 66721
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   City          69016 non-null  object        
 1   Beach         69016 non-null  object        
 2   Date          69016 non-null  datetime64[ns]
 3   Enterococcus  69016 non-null  int64         
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 2.6+ MB


In [4]:
df.loc[df['City']=='UBATUBA'].loc[df['Beach']=='PULSO']

,City,Beach,Date,Enterococcus
8303,UBATUBA,PULSO,2012-01-03,1
8304,UBATUBA,PULSO,2012-02-05,1
8305,UBATUBA,PULSO,2012-03-04,1
8306,UBATUBA,PULSO,2012-04-01,1
8307,UBATUBA,PULSO,2012-05-06,2
...,...,...,...,...
66655,UBATUBA,PULSO,2019-11-03,27
66656,UBATUBA,PULSO,2019-12-01,42
69027,UBATUBA,PULSO,2020-01-05,7
69028,UBATUBA,PULSO,2020-02-02,1


In [5]:
# selecionar praias com medições mensais
df2 = df.loc[df['City']=='GUARUJÁ'].loc[df['Beach']=='IPORANGA'] #75 linhas
df2 = df2.append(df.loc[df['City']=='IGUAPE'].loc[df['Beach']=='JURÉIA'])#105 linhas
df2 = df2.append(df.loc[df['City']=='ILHA COMPRIDA'].loc[df['Beach']=='BALNEÁRIO ADRIANA'])#94 linhas
df2 = df2.append(df.loc[df['City']=='ILHA COMPRIDA'].loc[df['Beach']=='CENTRO'])#102 Linhas
df2 = df2.append(df.loc[df['City']=='ILHA COMPRIDA'].loc[df['Beach']=='PONTAL'])#102 linhas
df2 = df2.append(df.loc[df['City']=='ILHA COMPRIDA'].loc[df['Beach']=='PRAINHA (BALSA)'])#102 Linhas
df2 = df2.append(df.loc[df['City']=='UBATUBA'].loc[df['Beach']=='PRUMIRIM'])#97 linhas
df2 = df2.append(df.loc[df['City']=='UBATUBA'].loc[df['Beach']=='PULSO'])#96 Linhas
df2

,City,Beach,Date,Enterococcus
16688,GUARUJÁ,IPORANGA,2014-01-05,133
16689,GUARUJÁ,IPORANGA,2014-02-02,45
16690,GUARUJÁ,IPORANGA,2014-03-02,1
16691,GUARUJÁ,IPORANGA,2014-04-06,1
16692,GUARUJÁ,IPORANGA,2014-05-04,1
...,...,...,...,...
66655,UBATUBA,PULSO,2019-11-03,27
66656,UBATUBA,PULSO,2019-12-01,42
69027,UBATUBA,PULSO,2020-01-05,7
69028,UBATUBA,PULSO,2020-02-02,1


In [6]:
n_steps = 4
for cidade in df2.City.unique():
    #print(df2.loc[df2['City']==cidade].Beach.unique())
    for praia in df2.loc[df2['City']==cidade].Beach.unique():
        #print(praia)
        df_beach = df.loc[df['City']==cidade].loc[df['Beach']==praia][['Date','Enterococcus']]
        df_beach.columns = ['ds', 'y']
        df_beach.set_index('ds', inplace=True)
        X, y = split_sequence(df_beach.values, n_steps)
        n_features = 1
        X = X.reshape((X.shape[0], X.shape[1], n_features))
        model = Sequential()
        model.add(LSTM(104, activation='relu', return_sequences=True, input_shape=(n_steps, n_features)))
        model.add(LSTM(104, activation='relu'))
        model.add(Dense(1))
        model.compile(optimizer='adam', loss='mse')
        with tf.device('/GPU:0'): 
            R = model.fit(X, y, epochs=200, verbose=1)
        loss=round(R.history['loss'][-1])
        model.save('model'+cidade+'-'+praia+'-'+str(loss)+'.h5')
        print(cidade+'-'+praia+'-'+str(loss))

Train on 71 samples
Epoch 1/200
71/71 [==============================] - 2s 28ms/sample - loss: 799.8286
Epoch 2/200
71/71 [==============================] - 0s 330us/sample - loss: 754.8524
Epoch 3/200
71/71 [==============================] - 0s 351us/sample - loss: 723.3947
Epoch 4/200
71/71 [==============================] - 0s 351us/sample - loss: 724.9940
Epoch 5/200
71/71 [==============================] - 0s 365us/sample - loss: 702.5327
Epoch 6/200
71/71 [==============================] - 0s 367us/sample - loss: 681.2958
Epoch 7/200
71/71 [==============================] - 0s 379us/sample - loss: 669.5022
Epoch 8/200
71/71 [==============================] - 0s 351us/sample - loss: 660.2432
Epoch 9/200
71/71 [==============================] - 0s 337us/sample - loss: 645.6356
Epoch 10/200
71/71 [==============================] - 0s 344us/sample - loss: 627.6371
Epoch 11/200
71/71 [==============================] - 0s 337us/sample - loss: 608.0033
Epoch 12/200
71/71 [=============

71/71 [==============================] - 0s 337us/sample - loss: 189.7604
Epoch 96/200
71/71 [==============================] - 0s 379us/sample - loss: 188.8417
Epoch 97/200
71/71 [==============================] - 0s 365us/sample - loss: 190.4412
Epoch 98/200
71/71 [==============================] - 0s 365us/sample - loss: 191.5159
Epoch 99/200
71/71 [==============================] - 0s 337us/sample - loss: 189.7074
Epoch 100/200
71/71 [==============================] - 0s 323us/sample - loss: 187.4221
Epoch 101/200
71/71 [==============================] - 0s 365us/sample - loss: 185.1331
Epoch 102/200
71/71 [==============================] - 0s 323us/sample - loss: 183.5156
Epoch 103/200
71/71 [==============================] - 0s 366us/sample - loss: 185.1543
Epoch 104/200
71/71 [==============================] - 0s 323us/sample - loss: 181.2107
Epoch 105/200
71/71 [==============================] - 0s 337us/sample - loss: 183.1694
Epoch 106/200
71/71 [=============================

98/98 [==============================] - 0s 326us/sample - loss: 304.1991
Epoch 82/200
98/98 [==============================] - 0s 346us/sample - loss: 301.7680
Epoch 83/200
98/98 [==============================] - 0s 346us/sample - loss: 305.6520
Epoch 84/200
98/98 [==============================] - 0s 315us/sample - loss: 308.7805
Epoch 85/200
98/98 [==============================] - 0s 326us/sample - loss: 307.8408
Epoch 86/200
98/98 [==============================] - 0s 325us/sample - loss: 305.6200
Epoch 87/200
98/98 [==============================] - 0s 326us/sample - loss: 303.5203
Epoch 88/200
98/98 [==============================] - 0s 351us/sample - loss: 305.9680
Epoch 89/200
98/98 [==============================] - 0s 315us/sample - loss: 304.5924
Epoch 90/200
98/98 [==============================] - 0s 326us/sample - loss: 301.4818
Epoch 91/200
98/98 [==============================] - 0s 346us/sample - loss: 299.2686
Epoch 92/200
98/98 [==============================] - 0s

Epoch 67/200
90/90 [==============================] - 0s 288us/sample - loss: 1943.1505
Epoch 68/200
90/90 [==============================] - 0s 255us/sample - loss: 1784.9088
Epoch 69/200
90/90 [==============================] - 0s 266us/sample - loss: 1694.9667
Epoch 70/200
90/90 [==============================] - 0s 283us/sample - loss: 1681.9149
Epoch 71/200
90/90 [==============================] - 0s 255us/sample - loss: 1608.5652
Epoch 72/200
90/90 [==============================] - 0s 277us/sample - loss: 1612.3073
Epoch 73/200
90/90 [==============================] - 0s 305us/sample - loss: 1526.7739
Epoch 74/200
90/90 [==============================] - 0s 277us/sample - loss: 1518.5228
Epoch 75/200
90/90 [==============================] - 0s 255us/sample - loss: 1479.2315
Epoch 76/200
90/90 [==============================] - 0s 255us/sample - loss: 1456.3653
Epoch 77/200
90/90 [==============================] - 0s 254us/sample - loss: 1420.1278
Epoch 78/200
90/90 [============

Epoch 160/200
90/90 [==============================] - 0s 255us/sample - loss: 679.7792
Epoch 161/200
90/90 [==============================] - 0s 255us/sample - loss: 716.5942
Epoch 162/200
90/90 [==============================] - 0s 299us/sample - loss: 689.2331
Epoch 163/200
90/90 [==============================] - 0s 266us/sample - loss: 734.9687
Epoch 164/200
90/90 [==============================] - 0s 266us/sample - loss: 744.0153
Epoch 165/200
90/90 [==============================] - 0s 255us/sample - loss: 784.5084
Epoch 166/200
90/90 [==============================] - 0s 288us/sample - loss: 733.7467
Epoch 167/200
90/90 [==============================] - 0s 277us/sample - loss: 762.1436
Epoch 168/200
90/90 [==============================] - 0s 255us/sample - loss: 692.1853
Epoch 169/200
90/90 [==============================] - 0s 255us/sample - loss: 695.3288
Epoch 170/200
90/90 [==============================] - 0s 244us/sample - loss: 696.4950
Epoch 171/200
90/90 [===========

...

98/98 [==============================] - 0s 377us/sample - loss: 721.3026
Epoch 130/200
98/98 [==============================] - 0s 325us/sample - loss: 657.2451
Epoch 131/200
98/98 [==============================] - 0s 295us/sample - loss: 682.7225
Epoch 132/200
98/98 [==============================] - 0s 328us/sample - loss: 726.3183
Epoch 133/200
98/98 [==============================] - 0s 326us/sample - loss: 696.0430
Epoch 134/200
98/98 [==============================] - 0s 377us/sample - loss: 661.4836
Epoch 135/200
98/98 [==============================] - 0s 326us/sample - loss: 639.3951
Epoch 136/200
98/98 [==============================] - 0s 315us/sample - loss: 618.2580
Epoch 137/200
98/98 [==============================] - 0s 305us/sample - loss: 604.5516
Epoch 138/200
98/98 [==============================] - 0s 326us/sample - loss: 590.3808
Epoch 139/200
98/98 [==============================] - 0s 325us/sample - loss: 587.4546
Epoch 140/200
98/98 [=========================

98/98 [==============================] - 0s 326us/sample - loss: 4174.0906
Epoch 114/200
98/98 [==============================] - 0s 326us/sample - loss: 3916.9077
Epoch 115/200
98/98 [==============================] - 0s 382us/sample - loss: 3407.2655
Epoch 116/200
98/98 [==============================] - 0s 356us/sample - loss: 3191.2192
Epoch 117/200
98/98 [==============================] - 0s 324us/sample - loss: 3040.6183
Epoch 118/200
98/98 [==============================] - 0s 336us/sample - loss: 2729.5633
Epoch 119/200
98/98 [==============================] - 0s 326us/sample - loss: 3157.8201
Epoch 120/200
98/98 [==============================] - 0s 346us/sample - loss: 3058.6569
Epoch 121/200
98/98 [==============================] - 0s 336us/sample - loss: 2377.2391
Epoch 122/200
98/98 [==============================] - 0s 326us/sample - loss: 10109.8767
Epoch 123/200
98/98 [==============================] - 0s 346us/sample - loss: 7133.8346
Epoch 124/200
98/98 [=============

Epoch 98/200
93/93 [==============================] - 0s 274us/sample - loss: 243.6171
Epoch 99/200
93/93 [==============================] - 0s 279us/sample - loss: 158.2373
Epoch 100/200
93/93 [==============================] - 0s 279us/sample - loss: 241.6914
Epoch 101/200
93/93 [==============================] - 0s 246us/sample - loss: 173.1859
Epoch 102/200
93/93 [==============================] - 0s 268us/sample - loss: 192.2440
Epoch 103/200
93/93 [==============================] - 0s 253us/sample - loss: 177.6739
Epoch 104/200
93/93 [==============================] - 0s 268us/sample - loss: 167.5397
Epoch 105/200
93/93 [==============================] - 0s 247us/sample - loss: 150.1210
Epoch 106/200
93/93 [==============================] - 0s 262us/sample - loss: 130.8138
Epoch 107/200
93/93 [==============================] - 0s 268us/sample - loss: 166.5745
Epoch 108/200
93/93 [==============================] - 0s 257us/sample - loss: 193.6013
Epoch 109/200
93/93 [=============

92/92 [==============================] - 0s 256us/sample - loss: 150.3179
Epoch 85/200
92/92 [==============================] - 0s 260us/sample - loss: 159.4538
Epoch 86/200
92/92 [==============================] - 0s 260us/sample - loss: 141.3843
Epoch 87/200
92/92 [==============================] - 0s 282us/sample - loss: 143.6816
Epoch 88/200
92/92 [==============================] - 0s 239us/sample - loss: 136.7234
Epoch 89/200
92/92 [==============================] - 0s 249us/sample - loss: 133.1735
Epoch 90/200
92/92 [==============================] - 0s 249us/sample - loss: 129.7935
Epoch 91/200
92/92 [==============================] - 0s 282us/sample - loss: 135.2955
Epoch 92/200
92/92 [==============================] - 0s 244us/sample - loss: 126.0153
Epoch 93/200
92/92 [==============================] - 0s 249us/sample - loss: 128.1660
Epoch 94/200
92/92 [==============================] - 0s 271us/sample - loss: 121.0105
Epoch 95/200
92/92 [==============================] - 0s

Epoch 178/200
92/92 [==============================] - 0s 260us/sample - loss: 68.7621
Epoch 179/200
92/92 [==============================] - 0s 249us/sample - loss: 76.0328
Epoch 180/200
92/92 [==============================] - 0s 260us/sample - loss: 68.4808
Epoch 181/200
92/92 [==============================] - 0s 271us/sample - loss: 70.9490
Epoch 182/200
92/92 [==============================] - 0s 293us/sample - loss: 69.7648
Epoch 183/200
92/92 [==============================] - 0s 238us/sample - loss: 71.5718
Epoch 184/200
92/92 [==============================] - 0s 260us/sample - loss: 65.9814
Epoch 185/200
92/92 [==============================] - 0s 293us/sample - loss: 64.9280
Epoch 186/200
92/92 [==============================] - 0s 271us/sample - loss: 64.2950
Epoch 187/200
92/92 [==============================] - 0s 260us/sample - loss: 62.8287
Epoch 188/200
92/92 [==============================] - 0s 282us/sample - loss: 62.3408
Epoch 189/200
92/92 [======================